# Neural Network from Scratch

This notebook builds neural-network mechanics progressively—from one neuron and a weighted sum to forward propagation, loss, gradients, backpropagation, a multi-layer perceptron, and the equivalent Keras workflow.

## 1. Core Equations

For one neuron:

- Weighted sum: $z = w · x + b$
- Sigmoid activation: $a = \sigma(z) = 1 / (1 + e^{-z})$
- Single-example squared error: $L = (a-y)^2$
- Gradient-descent update: $\theta_{new} = \theta_{old} - \eta \nabla_\theta L$

The forward pass produces a prediction. Backpropagation applies the chain rule from the loss back through each operation to calculate how every parameter should change.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 42

## 2. Single Neuron: Weighted Sum and Activation

The inputs and weights contribute to a scalar weighted sum. Sigmoid then maps that value to a number between 0 and 1.

In [ ]:
x = np.array([0.6, 0.8])
w = np.array([0.4, -0.2])
b = 0.1
y = 1.0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward_single(x, w, b):
    z = np.dot(w, x) + b
    a = sigmoid(z)
    return z, a

In [1]:
z, a = forward_single(x, w, b)
print(f"Weighted sum z: {float(z):.4f}")
print(f"Activation a: {float(a):.4f}")

Weighted sum z: 0.1800
Activation a: 0.5449


`z` is the neuron's raw weighted sum; `a` is the activated prediction. The weights `w` and bias `b` are the learnable parameters.

## 3. Loss and Gradients

The loss measures prediction error. For this single example, the chain rule follows
`dL/da → da/dz → dL/dz → dW, db`.

In [ ]:
def mse_loss(a, y):
    return (a - y) ** 2

def backward_single(x, y, a):
    dL_da = 2 * (a - y)
    da_dz = a * (1 - a)
    dL_dz = dL_da * da_dz
    dW = dL_dz * x
    db = dL_dz
    return dW, db

In [1]:
loss = mse_loss(a, y)
dW, db = backward_single(x, y, a)

print(f"Loss: {float(loss):.4f}")
print("dW:", dW)
print(f"db: {float(db):.4f}")

Loss: 0.2071
dW: [-0.1354 -0.1806]
db: -0.2257


## 4. Gradient Descent

A gradient gives the local direction and magnitude of change in the loss. Subtracting a scaled gradient moves the parameters toward lower loss.

In [1]:
learning_rate = 0.1
w_new = w - learning_rate * dW
b_new = b - learning_rate * db

z_new, a_new = forward_single(x, w_new, b_new)
new_loss = mse_loss(a_new, y)

print(f"Old loss: {float(loss):.4f}")
print(f"New loss: {float(new_loss):.4f}")
print("Updated weights:", w_new)
print(f"Updated bias: {float(b_new):.4f}")

Old loss: 0.2071
New loss: 0.1971
Updated weights: [ 0.4135 -0.1819]
Updated bias: 0.1226


The new loss is lower, confirming that this gradient step moved the prediction in the intended direction.

## 5. Multi-Layer Perceptron: Forward Propagation

The same operations extend to matrices. This network has two inputs, two hidden neurons, and one output neuron.

In [ ]:
x = np.array([1.0, 0.0])
y = 1.0

W1 = np.array([[0.1, 0.3], [0.2, 0.4]])
b1 = np.array([0.1, 0.1])
W2 = np.array([[0.5], [0.6]])
b2 = np.array([0.1])

def forward_mlp(x, W1, b1, W2, b2):
    z1 = x @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    y_hat = sigmoid(z2)
    cache = {"x": x, "a1": a1, "y_hat": y_hat}
    return y_hat, cache

In [1]:
y_hat, cache = forward_mlp(x, W1, b1, W2, b2)
print("Hidden activation:", cache["a1"])
print("Prediction:", y_hat)

Hidden activation: [0.5498 0.5987]
Prediction: [0.6757]


## 6. Backpropagation Through the MLP

Backpropagation starts at the output because the loss is calculated from the output prediction. The output error is propagated through `W2`, then multiplied by the hidden activation derivative to obtain the first-layer gradients.

In [ ]:
def backward_mlp(y, cache, W2):
    x = cache["x"]
    a1 = cache["a1"]
    y_hat = cache["y_hat"]

    delta2 = 2 * (y_hat - y) * y_hat * (1 - y_hat)
    dW2 = np.outer(a1, delta2)
    db2 = delta2

    da1 = delta2 @ W2.T
    delta1 = da1 * a1 * (1 - a1)
    dW1 = np.outer(x, delta1)
    db1 = delta1
    return dW1, db1, dW2, db2

In [1]:
dW1, db1, dW2, db2 = backward_mlp(y, cache, W2)
print("dW1:", dW1)
print("db1:", db1)
print("dW2:", dW2)
print("db2:", db2)

dW1: [[-0.0176 -0.0205]
 [-0.     -0.    ]]
db1: [-0.0176 -0.0205]
dW2: [[-0.0781]
 [-0.0851]]
db2: [-0.1421]


## 7. The Same Workflow in Keras

Keras performs the same weighted sums, activations, loss calculation, automatic differentiation, and parameter updates. The following small binary classifier connects the manual implementation to the framework API.

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)

X_train = np.array([
    [0.10, 0.20], [0.20, 0.30], [0.30, 0.25], [0.40, 0.35],
    [0.55, 0.60], [0.65, 0.70], [0.75, 0.65], [0.85, 0.90],
], dtype=np.float32)
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=np.float32)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(3, activation="sigmoid"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.1),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(X_train, y_train, epochs=300, verbose=0)
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")

## 8. Non-Linear Classification with Keras

The final example applies a two-hidden-layer MLP to `make_moons`, a dataset that cannot be separated by a single straight decision boundary. Training data is used to fit the scaler; validation guides training; the test split is evaluated once afterward.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_task, y_task = make_moons(n_samples=500, noise=0.18, random_state=SEED)

X_train_task, X_test_task, y_train_task, y_test_task = train_test_split(
    X_task, y_task, test_size=0.20, random_state=SEED, stratify=y_task
)
X_train_task, X_val_task, y_train_task, y_val_task = train_test_split(
    X_train_task, y_train_task, test_size=0.20, random_state=SEED, stratify=y_train_task
)

scaler = StandardScaler()
X_train_task = scaler.fit_transform(X_train_task)
X_val_task = scaler.transform(X_val_task)
X_test_task = scaler.transform(X_test_task)

In [ ]:
model_task = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model_task.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history_task = model_task.fit(
    X_train_task,
    y_train_task,
    validation_data=(X_val_task, y_val_task),
    epochs=100,
    batch_size=32,
    verbose=0,
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_task.history["loss"], label="Training loss")
plt.plot(history_task.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("Non-Linear Classifier Learning Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

test_loss, test_accuracy = model_task.evaluate(X_test_task, y_test_task, verbose=0)
probabilities = model_task.predict(X_test_task, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("First 10 predictions:", predictions[:10])

## Key Takeaways

- A neuron combines inputs through a weighted sum and activation.
- Loss converts prediction quality into an optimization objective.
- Gradients quantify how parameter changes affect that loss.
- Backpropagation applies the chain rule from the output toward earlier layers.
- An MLP learns intermediate representations by stacking these operations.
- Keras automates the same mathematics while preserving the define → compile → fit → evaluate workflow.